# 第12章 部品を組み立てる：Transformer

話し合い（Attention）と、ひとりで考える時間（MLP）を、何層も積み重ねる

## このノートブックの使い方・学習目標

- **教科書のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/12-transformer.html
- **エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **復習・理解度管理**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html)
- **この章でつくるもの**：GPTそのもの。マルチヘッドAttention・フィードフォワード・残差接続・LayerNormを組み合わせたミニGPTのクラス（第13章以降で使う `minigpt.py` と同じコード）
- **身につく数学・理論**：マルチヘッドAttention（分業） ／ フィードフォワード層（各位置で考える） ／ 残差接続（情報と勾配の高速道路） ／ Pre-LN（正規化してから処理する） ／ GPTの全体構造とパラメータの内訳 ／ まず1バッチで過学習させるテスト
- **新しく使うPython・PyTorch**：`nn.ModuleList` と `nn.Sequential` ／ `torch.cat` でつなげる ／ `self.apply` で全部品を初期化 ／ `named_parameters` で中身を調べる

## 実行環境・リソースの目安

| 項目 | 目安 |
|---|---|
| 推奨ランタイム | Google Colab（CPUでOK） |
| 所要時間目安 | 2時間 |
| 難易度 | ★★★★☆ |

> **✅ CPUでそのまま実行できます**：上のセルから順に再生ボタン（▶）または `Shift + Enter` で実行してください。

> **🎯 この章でつくるもの**
>
> 第11章のAttentionヘッドを部品にして、**GPT**を組み立てます。この章で書くクラスは、第13章で漱石の小説を学習させ、第15章では本物のGPT-2の重みを読み込むことになる、あなたのGPTの本体です。部品の1つ1つに「なぜそれが必要か」を確かめながら進みます。

> **🔗 これまでの復習とつながり：これまでの復習（第9・10・11章とのつながり）**
>
> この章で組み立てる GPT は、これまでに作ってきた3つの部品の合体ロボです。
>
> - **第9章（`nn.Module` と MLP）**：部品をクラスにまとめて `forward` を書く作法と、「全結合層 → 活性化関数 → 全結合層」の MLP（ここでは `FeedForward` として再登場）。
> - **第10章（`LayerNorm`・`Dropout`・初期化診察）**：活性化のばらつきを整える `LayerNorm`、過学習を防ぐ `Dropout`、そして初期損失が $\log V$ 付近か確かめる健康診断。
> - **第11章（1ヘッドの Self-Attention）**：`Head` クラス（Query・Key・Value・$\sqrt{d}$・因果マスク）、位置埋め込み `pos_emb`、そして「足し込む」残差接続の予告。

**ミナト**：Transformerって、「変形ロボット」みたいな名前ですよね。

**ボンゴ先生**：英語の transform は「変換する」だ。入力の文字列を、少しずつ「次の文字を当てやすい表現」に変換していく装置、という意味だよ。……まあ、合体ロボットっぽさはある。小さな部品を組み合わせて、巨大なものを作るからね。今日はキミが組み立て係だ。

## 準備

🔎 **実行前に確認**：コードの各行が何を受け取り、何を作るか確認してから ▶ を押してください。

In [ ]:
import re, math, urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F

def load_soseki(file_id="773_ruby_5968"):
    url = ("https://raw.githubusercontent.com/aozorahack/aozorabunko_text/0984f7dc18a2270bdfd2ff99bf14fd1b3ebb4f65/"
           f"cards/000148/files/{file_id}/{file_id}.txt")
    try:
        with urllib.request.urlopen(url, timeout=30) as res:
            text = res.read().decode("cp932")
    except Exception as e:
        raise RuntimeError(f"青空文庫の取得に失敗しました（時間をおいて再実行するか付録Eを参照）: {e}") from e
    text = text.split("-------------------------------------------------------")[2]
    text = text.split("底本：")[0]
    text = re.sub(r"《.*?》", "", text)
    text = re.sub(r"※?［＃.*?］", "", text)
    text = text.replace("｜", "").replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

text = load_soseki()
chars = sorted(set(text))
V = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text])
n = int(len(data) * 0.9)
train_data, val_data = data[:n], data[n:]

def get_batch(data, batch_size, T, g):
    ix = torch.randint(0, len(data) - T, (batch_size,), generator=g)
    x = torch.stack([data[i:i + T] for i in ix])
    y = torch.stack([data[i + 1:i + T + 1] for i in ix])
    return x, y

## 全体図

先に完成図を見ておきましょう。下から上に、データが流れていきます。

![ミニGPTの全体図。文字のIDが埋め込みになり、LayerNorm→マルチヘッドAttention→足し算、LayerNorm→フィードフォワード→足し算のブロックをN回通り、最後に出力層で次の文字の点数になる](https://sakunokyumei.github.io/llm-research-textbook/img/transformer.svg)

部品は5種類だけです。

1. **Head**（第11章で作ったAttentionヘッド）
2. **MultiHeadAttention**：ヘッドを何個も並べたもの
3. **FeedForward**：各位置で「考える」小さなMLP（第9章の親戚）
4. **Block**：2と3を、LayerNormと残差接続でつないだもの
5. **GPT**：埋め込み＋Blockを何層も＋出力層

順番に作っていきましょう。

#### 📝 確認問題（`ch12-q01` · Lv.1 · ミニGPTを構成する5部品とデータの流れ）

ミニGPT（`code/minigpt.py`）の中で、入力トークンID列 `idx` が「次の文字の点数 `logits`」になるまでの正しい順番はどれですか？

- **a.** 文字埋め込み → Transformer Block を 1 回だけ通過 → 各トークンに再び `tok_emb` の転置を掛け算して最終ロジットを出力
- **b.** 出力線形層（`lm_head`）→ 最終 `LayerNorm`（`ln_f`）→ Transformer Block を `n_layer` 回くり返す → 文字埋め込み＋位置埋め込み
- **c.** 文字埋め込み → `softmax` で確率化 → 位置ごとに `FeedForward` を `n_layer` 回くり返す → 出力線形層（Attention は使わない）
- **d.** 文字埋め込み＋位置埋め込み → Dropout → Transformer Block を `n_layer` 回くり返す → 最終 `LayerNorm`（`ln_f`）→ 出力線形層（`lm_head`）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：入力の整数 ID 列が、最初に何へ変換され、真ん中で何が何層も繰り返され、出口でどうやって語彙数 `V` の点数になるか流れを追いましょう。
- **ヒント 2**：入口は `tok_emb + pos_emb` と Dropout、本体は `Block` を `n_layer` 層、出口は `ln_f` と `lm_head` の順番です。
- **正解**：`d`

**考え方**：GPTは「埋め込み（入口）→ Block×N（話し合い＋考えるの反復）→ 最終LayerNorm＋出力層（出口）」という驚くほどシンプルな1本道でできています。

1. 入口：`self.drop(self.tok_emb(idx) + self.pos_emb(pos))`。
1. 本体：`self.blocks(x)`（`Block` を `n_layer` 個直列につないだ `nn.Sequential`）。
1. 出口：`self.lm_head(self.ln_f(x))` で `[B, T, vocab_size]` のロジットを得る。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/12-transformer.html#ex-ch12-q01

</details>

## 部品1：Head（おさらい）

第11章のHeadに、ドロップアウト（第10章）を1行足しただけです。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
class Head(nn.Module):
    """Attentionヘッド1つ（第11章）"""

    def __init__(self, n_embd, head_size, block_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        scores = q @ k.transpose(-2, -1) / math.sqrt(k.shape[-1])
        scores = scores.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        self.wei = F.softmax(scores, dim=-1)
        return self.dropout(self.wei) @ v

## 部品2：マルチヘッドAttention

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
class MultiHeadAttention(nn.Module):
    """ヘッドを何個も並べて、結果をつなげる"""

    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([Head(n_embd, head_size, block_size, dropout) for _ in range(n_head)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

mha = MultiHeadAttention(n_embd=64, n_head=4, block_size=8, dropout=0.0)
x = torch.randn(2, 8, 64)
print(mha(x).shape)

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
torch.Size([2, 8, 64])
```

</details>

> **💡 理論のポイント：ヘッドを増やして分業する**
>
> 1つのヘッドは、1種類の「注目の仕方」しかできません。でも文章を理解するには、「直前の文字」「文の主語」「かぎかっこの始まり」など、いろいろな観点で過去を見る必要があります。そこで、**小さなヘッドを何個も並べて**、それぞれに違う見方を学ばせます。
>
> 64個の数を4つのヘッドで分けると、各ヘッドは16個ずつ担当します（`head_size = n_embd // n_head`）。4つのヘッドの結果を `torch.cat` で横につなげると、また64個に戻ります。最後の `proj`（全結合層）で、4つの見方を混ぜ合わせます。
>
> 会議で、全員が同じ資料の同じ行を見ていたら意味がありませんよね。分担して違うところを読み、最後に持ち寄る。それがマルチヘッドです。第13章で、学習したヘッドたちが実際に役割分担している様子を見ます。

> **🐍 Python・PyTorchのポイント：ModuleList**
>
> `nn.ModuleList([...])` は、部品のリストをPyTorchに「これも全部モデルの部品だよ」と教えるための入れ物です。普通のPythonのリスト `[...]` に入れただけだと、PyTorchがパラメータを見つけられず、学習されません（`model.parameters()` に入らない）。気づきにくい落とし穴です。

<details><summary><b>基礎の補足：マルチヘッドAttentionの形トレースと「なぜ分業が得なのか？」</b>（クリックで開く）</summary>

`n_embd = 64`、`n_head = 4`（つまり `head_size = 64 // 4 = 16`）のとき、テンソルの形は次のように流れて元に戻ります。

| ステップ | テンソルの形 | 何をしているか |
|---|---|---|
| 入力 `x` | `[B, T, 64]` | 元の埋め込みベクトル（`n_embd = 64`） |
| 各ヘッド `h(x)` × 4個 | `[B, T, 16]` が 4本 | 4つのヘッドが独立に「別の観点」で過去から情報を集める（`hs = 16`） |
| `torch.cat(..., dim=-1)` | `[B, T, 64]` | 4本の `[B, T, 16]` を最後の軸で横につなぎ、`4 × 16 = 64` に戻す |
| `self.proj(out)` | `[B, T, 64]` | 全結合層で4つのヘッドの見方を混ぜ合わせて残差ストリームへ返す |

しかも嬉しいことに、1つの大きいヘッド（`head_size = 64`）を4つの小さいヘッド（`head_size = 16` × 4）に分けても、Q・K・V のパラメータ合計は $4 \times (3 \times 64 \times 16) = 3 \times 64^2$ で**1個も増えません**。同じパラメータ予算のまま、複数の注目パターン（直前の文字、主語、括弧の対応など）を同時並行で追えるのがマルチヘッドの強みです。なお、最後に元の `n_embd` へ戻すため、必ず `n_embd % n_head == 0`（割り切れる設定）にします。

</details>

#### 📝 確認問題（`ch12-q02` · Lv.2 · マルチヘッドAttentionの head_size 計算（n_embd // n_head））

`n_embd = 384`、`n_head = 6` のとき、マルチヘッドAttention の1つのヘッドの大きさ（`head_size`）はいくつになりますか？

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：マルチヘッド Attention では、全体の埋め込み次元 `n_embd` をヘッド数 `n_head` で均等に分割します。
- **ヒント 2**：`head_size = n_embd // n_head` に `n_embd = 384`、`n_head = 6` を代入して割り算しましょう。
- **正解**：`64`

**考え方**：全体の埋め込み次元 `n_embd` を `n_head` 個のヘッドに均等に分け、各ヘッドの出力を最後に横につなげて元の `n_embd` に戻します。

1. `n_embd = 384`、`n_head = 6`。
1. `head_size = 384 // 6 = 64`。
1. 6個のヘッド出力 `[B, T, 64]` を `dim=-1` で連結すると `[B, T, 384]` に戻る。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/12-transformer.html#ex-ch12-q02

</details>

#### 📝 確認問題（`ch12-q03` · Lv.2 · マルチヘッドの分業・torch.cat(dim=-1)・self.proj の役割）

`MultiHeadAttention.forward` では `out = torch.cat([h(x) for h in self.heads], dim=-1)` のあとに `self.proj(out)` を通しています。`torch.cat(..., dim=-1)` と `self.proj` の役割の組み合わせとして正しいものはどれですか？

- **a.** `torch.cat(..., dim=-1)` でバッチ軸方向に並べて文章の数 `B` をヘッド数倍に増やし、`self.proj` で最も点数の高い 1 文章だけ選んで残りを切り捨てる
- **b.** `torch.cat(..., dim=-1)` の時点で全ヘッドの情報は完全に混ざり合っているため、`self.proj` はコードの見栄えを整えるだけの飾りであり削除しても同じである
- **c.** `torch.cat(..., dim=-1)` で全ヘッドの出力テンソルの要素ごとの平均を計算し、`self.proj` で未来の位置を `-inf` にして因果マスクを適用する
- **d.** `torch.cat(..., dim=-1)` で各ヘッドの出力 `[B, T, hs]` を特徴次元方向に連結して `[B, T, n_embd]` に戻し、`self.proj`（線形層）で複数ヘッドの見方を混ぜ合わせる

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：4 人の担当者がそれぞれ 16 次元のメモを持ち寄って 1 列に並べた（`cat`）あと、最後の線形層 `self.proj` が何をしているか考えてみましょう。
- **ヒント 2**：`dim=-1` は末尾の特徴軸をつないで `[B, T, n_embd]` に戻す操作、`self.proj` は並べただけの各ヘッド情報を線形結合で統合する層です。
- **正解**：`d`

**考え方**：小さなヘッドを並列に走らせて特徴次元で連結すると、1つの大きいヘッドとほぼ同じ計算量・QKVパラメータ数のまま、複数の異なる注目パターン（直前、主語、括弧など）を同時に捉えられます。

1. 各ヘッド `h(x)` の出力は `[B, T, head_size]`。
1. `n_head` 個を `dim=-1` で連結すると `[B, T, n_head * head_size] = [B, T, n_embd]`。
1. `self.proj(out)` で各ヘッドの情報を線形結合して混ぜ合わせ、残差接続へ渡す。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/12-transformer.html#ex-ch12-q03

</details>

*(この節の追加問題 `ch12-q04` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/12-transformer.html で自動採点・復習できます)*

## 部品3：フィードフォワード（考える時間）

🔎 **実行前に確認**：コードの各行が何を受け取り、何を作るか確認してから ▶ を押してください。

In [ ]:
class FeedForward(nn.Module):
    """各位置で、ひとりで「考える」MLP"""

    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

> **💡 理論のポイント：話し合いと、ひとりで考える時間**
>
> Attentionは「文字どうしが情報をやりとりする」部品でした。いわば**話し合い**です。でも、話し合いで集めた情報を**消化する時間**も必要です。それがフィードフォワード層です。各位置で独立に（ほかの位置とは関係なく）、第9章と同じ「全結合層 → 活性化関数 → 全結合層」の小さなMLPを通します。
>
> 真ん中で一度 **4倍** の幅に広げるのが定番です（GPTの論文の設計）。広い作業机でいったん情報を広げて考え、また元の幅にまとめる、というイメージです。活性化関数には、GPTと同じ **GELU** を使います。
>
> 実は、モデルの「知識」の多く（「漱石」の次は「夏目」が来やすい、など）は、このフィードフォワード層に蓄えられていると考えられています（第25章の解釈可能性の話題）。

> **🐍 Python・PyTorchのポイント：Sequential**
>
> `nn.Sequential(A, B, C)` は、部品を**順番に通す**入れ物です。`self.net(x)` と書くと、`C(B(A(x)))` と同じ計算になります。流れ作業のベルトコンベアです。

<details><summary><b>基礎の補足：FFNの4倍拡張：形トレースとパラメータ数</b>（クリックで開く）</summary>

`n_embd` を $C$ とすると、`FeedForward` の内部ではテンソルの形がいったん **4倍** に広がってから元に戻ります。

- 入力：`[B, T, C]`
- 1層目 `Linear(C, 4C)` ＋ `GELU`：`[B, T, 4C]`（パラメータ数：$C \times 4C + 4C = 4C^2 + 4C$）
- 2層目 `Linear(4C, C)` ＋ `Dropout`：`[B, T, C]`（パラメータ数：$4C \times C + C = 4C^2 + C$）

合計は $8C^2 + 5C$ 個です。実は、1つの `Block` 全体のパラメータ（約 $12C^2$）のうち **約 2/3 がこの FeedForward 層**にあります（Multi-Head Attention は約 $4C^2$＝約 1/3）。広い作業机（$4C$）に大量の重みを置いて、話し合いで集めた文脈を各位置でしっかり加工・記憶しているのです。

</details>

#### 📝 確認問題（`ch12-q05` · Lv.2 · FeedForward 層の 4倍拡張・GELU・独立処理の意味）

`FeedForward` 層（`Linear(C, 4*C) → GELU → Linear(4*C, C) → Dropout`）の説明として**正しい**ものはどれですか？

- **a.** 文章中の全トークン位置どうしの内積を計算して注目重みを作り、過去の文字から必要な文脈情報を集めて混ぜ合わせる層である
- **b.** 中間の `GELU` 活性化関数を取り除いて 2 枚の線形層だけにしてしまっても、作業空間が 4 倍広いおかげで表現力はまったく変わらない
- **c.** 各トークン位置ごとに独立に適用され、一度次元を 4 倍（`4 * n_embd`）の広い作業空間へ広げて非線形変換（GELU）してから元の次元 `n_embd` に戻す
- **d.** 入力 `[B, T, C]` を受け取ると出力の形が `[B, T, 4*C]` へ拡張されたままになり、そのまま次の Transformer Block へ引き渡される

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`FeedForward` が「時間軸（トークン間）」と「特徴次元（各位置の内部）」のどちらに作用し、入口と出口の `shape` がどうなるか確認しましょう。
- **ヒント 2**：入口 `[B, T, C]` → 1層目＋GELU `[B, T, 4C]` → 2層目 `[B, T, C]` の順に各位置独立で変換し、残差接続で足せるよう元の次元 `C` に戻します。
- **正解**：`c`

**考え方**：Attention で集めた情報を、各位置ごとに広い中間表現（`4 * n_embd`）へ展開して非線形に組み替え、再び残差ストリームの幅 `n_embd` へ戻します。モデルの事実知識の多くはこの層に蓄えられます。

1. 入力：`[B, T, C]`。
1. 1層目 `Linear(C, 4C)` + `GELU`：`[B, T, 4C]`（広い空間で特徴を組み合わせる）。
1. 2層目 `Linear(4C, C)` + `Dropout`：`[B, T, C]`（元の幅に戻す）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/12-transformer.html#ex-ch12-q05

</details>

#### 📝 確認問題（`ch12-q06` · Lv.3 · FeedForward 層の正確なパラメータ数計算）

`n_embd = 16` のとき、`FeedForward(n_embd=16, dropout=0.1)` のパラメータ総数（バイアス込み）はいくつになりますか？

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`FeedForward` の学習パラメータは 2 枚の `nn.Linear`（`C → 4C` と `4C → C`、どちらもバイアスあり）だけです。`GELU` と `Dropout` は 0 個です。
- **ヒント 2**：1層目 `Linear(16, 64)` の `16 * 64 + 64` と、2層目 `Linear(64, 16)` の `64 * 16 + 16` を足し合わせましょう（公式 $8C^2 + 5C$）。
- **正解**：`2128`

**考え方**：`nn.Linear(in_f, out_f)` は重み `in_f * out_f` とバイアス `out_f` を持ちます。`GELU` と `Dropout` のパラメータは0個です。

1. `4 * n_embd = 64`。
1. 1層目 `Linear(16, 64)`：`16 * 64 + 64 = 1024 + 64 = 1088`。
1. 2層目 `Linear(64, 16)`：`64 * 16 + 16 = 1024 + 16 = 1040`。
1. 合計：`1088 + 1040 = 2128`（公式 $8 \times 16^2 + 5 \times 16 = 2048 + 80 = 2128$）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/12-transformer.html#ex-ch12-q06

</details>

## 部品4：Block（残差接続とLayerNorm）

🔎 **実行前に確認**：コードの各行が何を受け取り、何を作るか確認してから ▶ を押してください。

In [ ]:
class Block(nn.Module):
    """Transformerブロック：話し合い（Attention）→ 考える（FeedForward）"""

    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = MultiHeadAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

たった2行の `forward` ですが、ここにTransformerの設計の知恵が詰まっています。

> **💡 理論のポイント：残差接続：情報の高速道路**
>
> `x = x + self.attn(…)` の **`x +`** の部分が **残差接続**（residual connection）です。第11章でも1回出てきましたね。
>
> 部品の出力で `x` を**置き換える**のではなく、`x` に**足し込む**。こうすると、
>
> - **情報の高速道路（ベルトコンベア）**：元の情報（自分が何の文字か、など）が、何層通っても消えずに残る。残差ストリーム（residual stream）は「工場のベルトコンベア」のようなものです。コンベアの上を文章情報がまっすぐ流れ、各ブロックのAttention係やMLP係は、コンベアの荷物を覗き見て「追加の気づき（差分）」をそっと上に載せるだけです。コンベアそのものを上書きしないので、何十層重ねても最初の文字情報が消えません。
> - **勾配の高速道路**：誤差逆伝播で、勾配が足し算の道を通って、入口までまっすぐ届く（足し算の微分は1なので、勾配が弱まらない）。第10章の「深い層で勾配が消える」問題を、根本から解決する
>
> LLMの層を何十、何百と深くできるのは、この残差接続のおかげです。2015年の画像認識モデル **ResNet** で有名になった方法で、Transformerはこれを借りてきました。

> **💡 理論のポイント：位置埋め込みを「足し算」する理由**
>
> Attentionには「順序」という概念がありません。文字の並び順をシャッフルしても、位置の情報がないとまったく同じ計算になってしまいます。そこで、0番目、1番目、2番目……という位置ごとのベクトル（`pos_emb`）を用意し、文字のベクトル（`tok_emb`）に**そのまま足し算**します。「足し算して混ざってしまわないの？」と不安になりますが、高次元の空間では、文字の意味の向きと位置の向きを別々の方向（独立した軸）として両立して保持できるため、ちゃんと区別して学習されます。

> **💡 理論のポイント：Pre-LN：まず整えてから処理する**
>
> 各部品に入る**前に** LayerNorm をかけています（`self.attn(self.ln1(x))`）。これを **Pre-LN**（前置き正規化）と言います。2017年の元祖Transformerは「処理した後に」正規化していましたが、GPT-2以降は、学習が安定するPre-LNが主流です。残差ストリーム `x` 自体は正規化せず、各作業員が仕事をする前に、手元のコピーだけを整えるイメージです。

<details><summary><b>基礎の補足：残差接続の「勾配の高速道路」と Pre-Norm vs Post-Norm</b>（クリックで開く）</summary>

**① なぜ `x + sublayer(ln(x))` で勾配が消えないのか？**
残差接続の式 $y = x + F(x)$ を $x$ で微分すると、足し算の微分ルールから次の式になります。

$$\frac{\partial y}{\partial x} = 1 + \frac{\partial F(x)}{\partial x}$$

もし残差接続がない（$y = F(x)$）と、層を $L$ 回さかのぼるたびに微分の掛け算 $F_L' \cdots F_1'$ が起き、各層が $0.5$ なら 20 層で $0.5^{20} \approx 10^{-6}$ に消えてしまいます（勾配消失）。一方、残差接続があれば必ず **`+ 1` の直通ルート**が残るため、たとえ $F'(x)$ が小さくても、勾配が減衰せず入口までまっすぐ届きます。

**② Pre-Norm と Post-Norm の比較**

| 方式 | コードの形 | 特徴 |
|---|---|---|
| **Pre-Norm（GPT-2以降の主流）** | `x = x + sublayer(ln(x))` | 幹線道路 `x` に LayerNorm が挟まらず純粋な足し算を保てる。深い層でも学習が安定（最後に `ln_f` を1回入れる） |
| **Post-Norm（2017年元祖Transformer）** | `x = ln(x + sublayer(x))` | 毎層、幹線道路 `x` 自体が正規化を通るため、層を深くすると勾配が不安定になりやすい |

</details>

#### 📝 確認問題（`ch12-q07` · Lv.3 · 残差接続 x = x + sublayer(ln(x)) が勾配消失を防ぐ理由）

`Block.forward` の `x = x + self.attn(self.ln1(x))` から `x +` を消して `x = self.attn(self.ln1(x))` に変えると、特に層を深くしたときに何が困りますか？

- **a.** 残差接続が担当していた学習可能パラメータがすべて削除されるため、Transformer ブロック全体のパラメータ数がちょうど半分に減ってしまう
- **b.** 元の情報が毎層上書きされて失われやすく、さらに逆伝播の勾配が $1 + F'(x)$ の近道を失って毎層掛け算で弱まり、深い層の学習が進みにくくなる（勾配消失）
- **c.** 各 `Head` 内部に組み込まれている下三角の因果マスクが無効化されてしまい、学習時に未来のトークンをカンニングするバグが発生する
- **d.** 要素ごとの足し算が消えることでテンソルの形（shape）が `[B, T, C]` から `[B, T, 2*C]` へ変わってしまい、次の層の入力で必ず形エラーが起きる

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：残差接続 $y = x + F(x)$ と残差なし $y = F(x)$ をそれぞれ $x$ で微分したとき、連鎖律の掛け算の中にどんな項が残るか比べてみましょう。
- **ヒント 2**：残差ありでは $\frac{\partial y}{\partial x} = 1 + F'(x)$ に `+ 1` の直通ルートが残るため、層を深くしても勾配が掛け算で消えずに奥まで届きます。
- **正解**：`b`

**考え方**：残差接続 $y = x + F(x)$ では微分の式に必ず `+ 1` が残るため、たとえ途中の層の微分 $F'(x)$ が小さくても、勾配が入口まで減衰せずまっすぐ届く「勾配の高速道路」になります。

1. 残差なし：$y = F(x)$ → 層を $L$ 回通ると微分が $F_L' \cdots F_1'$ の掛け算になり、$0.5^{20} \approx 10^{-6}$ のように消える。
1. 残差あり：$y = x + F(x)$ → 微分は $1 + F'(x)$。`+ 1` の直通ルートを通って勾配が入口まで届く。
1. 順伝播でも元の情報（残差ストリーム）に差分だけをそっと上乗せできる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/12-transformer.html#ex-ch12-q07

</details>

#### 📝 確認問題（`ch12-q08` · Lv.3 · Pre-Norm と Post-Norm の違い（なぜ GPT-2 以降は Pre-Norm なのか））

現代のGPT（`Block`）では **Pre-Norm**（`x = x + self.attn(self.ln1(x))`）が使われ、2017年の元祖 Transformer の **Post-Norm**（`x = self.ln1(x + self.attn(x))`）とは LayerNorm の位置が違います。Pre-Norm の最大の利点はどれですか？

- **a.** Pre-Norm の位置で正規化しておけば過学習が完全に抑えられるため、ブロック内のすべての `Dropout` 層を削除して学習を高速化できる
- **b.** 残差ストリーム `x` の幹線道路上に LayerNorm を挟まず「純粋な足し算」のまま保てるため、深い層でも勾配が安定して学習しやすい
- **c.** Pre-Norm では各ブロックの出口で出力が必ず正規化されるため、`lm_head` の直前に置く最終 `ln_f` が不要になり構造が簡単になる
- **d.** Post-Norm では 1 ブロックに 4 個必要な `LayerNorm` を 2 個へ減らせるため、正規化層のパラメータ数がちょうど半分になる

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：Post-Norm（`ln(x + sublayer(x))`）と Pre-Norm（`x + sublayer(ln(x))`）で、残差ストリーム `x` の直通ルートに何が挟まるか比べてみましょう。
- **ヒント 2**：Pre-Norm は幹線道路 `x` を純粋な足し算のまま保つことで、勾配の高速道路を塞がずに深い層まで安定して学習できます。
- **正解**：`b`

**考え方**：Post-Norm では毎回残差ストリーム全体が LayerNorm を通るため、深い層では勾配の大きさが不安定になりやすいです。Pre-Norm は幹線道路を純粋な足し算に保つことで学習を劇的に安定させます。

1. Post-Norm：`x = ln(x + sublayer(x))` → 幹線道路 `x` の途中に毎回 `ln` が挟まる。
1. Pre-Norm：`x = x + sublayer(ln(x))` → 幹線道路 `x` は純粋な足し算のまま、部品への入力だけを整える。
1. 最後に `ln_f(x)` で整えてから `lm_head` へ渡す。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/12-transformer.html#ex-ch12-q08

</details>

## 部品5：GPT（全体）

最後に、全部をつなげます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab_size, block_size=128, n_embd=128, n_head=4, n_layer=4, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        x = self.blocks(x)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        if temperature <= 0:
            raise ValueError("temperature must be positive")
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]          # 窓に入る分だけ
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature      # 最後の位置の点数
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

> **🐍 Python・PyTorchのポイント：新しく出てきた書き方**
>
> - `nn.Sequential(*[Block(…) for _ in range(n_layer)])`：Blockを `n_layer` 個作ってリストにし、`*` でばらして `Sequential` に渡しています。Blockを何層も積み重ねる、の1行版です。
> - `self.apply(self._init_weights)`：モデルの中の**すべての部品**に対して `_init_weights` を実行します。GPT-2と同じく、重みを標準偏差0.02の正規分布で初期化します（第10章の初期化の話）。
> - `isinstance(module, nn.Linear)`：「`module` は `nn.Linear` の仲間（インスタンス）か？」を `True` / `False` で答える関数です。`(nn.Linear, nn.Embedding)` のように並べると「どちらかの仲間か？」になります。部品の種類ごとに、初期化の仕方を変えるために使っています。
> - `generate` は、第3章で自作した生成ループと同じです。違いは、入力を最後の `block_size` 文字に切り詰めること（位置の埋め込みは `block_size` 個しかないので）と、上位 `top_k` 個以外を候補から外せること（考え方は第16章で詳しく）。`v, _ = torch.topk(logits, k)` で上位k個の点数 `v` を大きい順に取り出し、`v[:, [-1]]`（その一番最後＝k位の点数。`[-1]` を `[ ]` で囲むと、形が `[B, 1]` のまま残ります）より小さい点数を、`-inf`（softmaxで確率0になる）に置き換えています。

#### 📝 確認問題（`ch12-q09` · Lv.2 · self.apply による初期化（std=0.02）と generate 時の block_size 切り詰め）

`GPT.generate` の先頭に `idx_cond = idx[:, -self.block_size:]` という1行があるのはなぜですか？

- **a.** 前の生成ステップで掛けたサンプリング温度パラメータの影響を打ち消し、次の文字予測の前に温度を必ず `1.0` へリセットするため
- **b.** 文章全体の話題を見失わないよう、生成が進んでも毎回必ず文章の先頭にある最初の `block_size` 文字だけを固定して見続けさせるため
- **c.** 長さが `block_size` を超えると `pos_emb` や `tril` の範囲外になるため、直近 `block_size` 文字だけ切り出すため
- **d.** 推論時のメモリ使用量を最小限に抑えるため、複数文章をまとめたバッチサイズ `B` の軸を切り詰めて常に `1` へ減らすため

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`self.pos_emb = nn.Embedding(block_size, n_embd)` や `self.tril` は最大で何文字分までの位置しか用意されていないでしょうか？
- **ヒント 2**：スライス `[:, -self.block_size:]` は末尾から最大 `block_size` 文字を切り出します。これによりどれだけ長く生成しても `pos_emb` や `tril` の範囲外エラーを防げます。
- **正解**：`c`

**考え方**：固定長の位置埋め込みを持つ GPT では、入力長が `block_size` を1文字でも超えると `pos_emb(torch.arange(T))` で `IndexError` が起きます。スライス `[:, -self.block_size:]` により、どれだけ長く生成しても常に最新の窓幅に収まります。

1. 生成前は長さ `T <= block_size` でも、`max_new_tokens` 文字生成すると `T > block_size` になり得る。
1. `idx[:, -self.block_size:]` は長さが `block_size` 未満ならそのまま、超えたら末尾 `block_size` 文字だけを取る。
1. これによって `pos_emb` と `tril` のサイズ内に常に収まる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/12-transformer.html#ex-ch12-q09

</details>

#### 📝 確認問題（`ch12-q10` · Lv.3 · コード予測：generate 内の top_k フィルタリングの動き）

`GPT.generate` の `top_k` 処理と同じロジックを純Pythonで書きました。次のコードを実行すると何が表示されますか？

```python
logits = [1.0, 5.0, 2.0, 4.0, 0.5]
k = 2
thresh = sorted(logits, reverse=True)[k - 1]
filtered = [x if x >= thresh else float("-inf") for x in logits]
print(filtered)
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`sorted(logits, reverse=True)[k - 1]` は大きいほうから `k` 番目の値をしきい値 `thresh` にします。それ未満の要素が何に置き換わるか見ましょう。
- **ヒント 2**：`k = 2` なので大きい順の上位 2 個は `5.0` と `4.0`、`thresh = 4.0` です。元の並び順のまま `4.0` 未満を `-inf` に置き換えたリストを書き出します。
- **期待される出力**：`[-inf, 5.0, -inf, 4.0, -inf]`

**考え方**：`top_k = 2` は上位2位の点数 `thresh` 未満の候補をすべて `-inf` に置き換えます。そのあと softmax をかけると上位2個以外が確率 `0.0` になり、変な低確率トークンを引く暴走を防げます。

1. `sorted(logits, reverse=True)` → `[5.0, 4.0, 2.0, 1.0, 0.5]`。
1. `thresh = [k - 1]`（インデックス1）→ `4.0`。
1. `4.0` 未満（`1.0, 2.0, 0.5`）が `-inf` に置換される。
1. `[-inf, 5.0, -inf, 4.0, -inf]` が出力される。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/12-transformer.html#ex-ch12-q10

</details>

## 動かしてみる

CPUでも動く小さめの設定で、GPTを作ってみましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
torch.manual_seed(0)
model = GPT(V, block_size=64, n_embd=64, n_head=4, n_layer=2, dropout=0.0)
total = sum(p.numel() for p in model.parameters())
print(f"パラメータの数：{total:,}")
groups = {"埋め込み（文字・位置）": 0, "Transformerブロック": 0, "最後のLayerNorm＋出力層": 0}
for name, p in model.named_parameters():
    if name.startswith(("tok_emb", "pos_emb")):
        groups["埋め込み（文字・位置）"] += p.numel()
    elif name.startswith("blocks"):
        groups["Transformerブロック"] += p.numel()
    else:
        groups["最後のLayerNorm＋出力層"] += p.numel()
for k, v in groups.items():
    print(f"  {k}：{v:,}（{v / total:.0%}）")   # :, は3桁ごとのカンマ、:.0% はパーセント表示

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
パラメータの数：369,677
  埋め込み（文字・位置）：136,000（37%）
  Transformerブロック：99,584（27%）
  最後のLayerNorm＋出力層：134,093（36%）
```

</details>

まだ小さなモデルなので、語彙（2061文字）に関わる埋め込みと出力層がパラメータの大半を占めています。Blockを増やしたり、`n_embd` を大きくしたりすると、Transformerブロックの割合が増えていきます。

第10章の診察その1、「最初の損失を見る」もやっておきましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
g = torch.Generator().manual_seed(0)
x, y = get_batch(train_data, 8, 64, g)
_, loss = model(x, y)
print("最初の損失：", round(loss.item(), 3), " log V：", round(math.log(V), 3))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
最初の損失： 7.656  log V： 7.631
```

</details>

$\log V$ とほぼ一致。初期化は健康です。

<details><summary><b>基礎の補足：GPTのパラメータ数を手で検算する公式</b>（クリックで開く）</summary>

上の実測値 `369,677` は、語彙数 $V$・窓幅 $T$・埋め込み次元 $C$・層数 $L$ から **1個の狂いもなく** 手で検算できます。

| グループ | 部品の内訳 | 公式 | `V=2061, T=64, C=64, L=2` の値 |
|---|---|---|---|
| 埋め込み | `tok_emb` ($VC$) ＋ `pos_emb` ($TC$) | $(V + T)C$ | $(2061 + 64) \times 64 = \mathbf{136,000}$ |
| Transformerブロック（1層分） | `ln1,ln2` ($4C$) ＋ `attn` ($4C^2+C$) ＋ `ffwd` ($8C^2+5C$) | $12C^2 + 10C$ | $12 \times 64^2 + 10 \times 64 = 49,792$ |
| ブロック全体（$L$ 層分） | 上の1層分 × $L$ | $L(12C^2 + 10C)$ | $2 \times 49,792 = \mathbf{99,584}$ |
| 最終LayerNorm＋出力層 | `ln_f` ($2C$) ＋ `lm_head` ($CV + V$) | $2C + CV + V$ | $128 + 131,904 + 2061 = \mathbf{134,093}$ |
| **合計** | — | — | **369,677**（実測と完全一致！） |

下の確認問題で、この計算と Block 内のテンソル shape トレースを純Pythonで自力実装してみましょう！

</details>

#### 📝 確認問題（`ch12-q11` · Lv.3 · GPTのパラメータ内訳とスケール時の変化）

小さい設定（`V=2061, n_embd=64, n_layer=2`）では埋め込みと出力層が全体の約73%を占め、Transformerブロックは27%だけでした。ここから `n_embd` や `n_layer` を大きくしていくと（例：GPT-2や大規模LLM）、パラメータの割合はどう変わりますか？

- **a.** 埋め込み・出力層は $O(V \cdot C)$ で増える一方、ブロック群は $O(L \cdot C^2)$ で増えるため、モデルが大きいほどブロックが圧倒的大多数を占めるようになる
- **b.** 各 Transformer ブロックは全層でまったく同じ 1 組の重みを共有して使い回す設計になっているため、`n_layer` をいくら増やしてもブロックのパラメータ総数は変わらない
- **c.** 文脈を長く処理するための位置埋め込み `pos_emb` が層数と次元の両方の 2 乗に比例して急増するため、大規模モデルでは `pos_emb` が全体の 90% 以上を占めるようになる
- **d.** どれだけ `n_embd` や `n_layer` を大きくしても、埋め込み・出力層が約 73%・Transformer ブロックが約 27% という構成比率はモデルの構造上ずっと一定に保たれる

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：埋め込み＋出力層のパラメータ数（約 $2VC$）と、Transformer ブロック全体のパラメータ数（約 $12 L C^2$）について、$C$ と $L$ を増やしたときの増え方を比べましょう。
- **ヒント 2**：語彙数 $V$ はあまり変わらない一方、ブロック群は $C^2$ と層数 $L$ の積で急増するため、大きなモデルほどブロックが全体の大部分を占めます。
- **正解**：`a`

**考え方**：語彙数 $V$ はモデルを大きくしてもほぼ一定ですが、$C$（`n_embd`）と $L$（`n_layer`）は大きく増えるため、大規模LLMではパラメータの大部分（80〜95%以上）が Transformer ブロック本体に集中します。

1. 埋め込み＋出力層：約 $2 V C$（$C$ の1乗に比例）。
1. Transformer ブロック群：約 $12 L C^2$（$C$ の2乗と層数 $L$ に比例）。
1. 例：第12章（$C=64, L=2$）ではブロック27% → 第13章（$C=128, L=4$）では約45% → 第13章 `train_big`（$C=384, L=6$）では約82%！

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/12-transformer.html#ex-ch12-q11

</details>

#### 📝 確認問題（`ch12-q12` · Lv.4 · 【自力実装】GPTの設定 (V, T, C, n_layer) から正確なパラメータ総数を計算）

PyTorch を使わず純Pythonで、第12章の `GPT(vocab_size, block_size, n_embd, n_head, n_layer)`（`code/minigpt.py` と同一構造）の**正確なパラメータ総数**（整数）を返す関数 `count_gpt_params(V, block_size, n_embd, n_layer)` を実装し、本文の実測値 `369,677` などと完全一致することを検証してください。

**ひな形コード：**
```python
def count_gpt_params(V, block_size, n_embd, n_layer):
    C = n_embd
    # 1. emb: tok_emb (V * C) + pos_emb (block_size * C)
    # 2. per_block: ln1+ln2 (4*C) + attn (4*C*C + C) + ffwd (8*C*C + 5*C)
    # 3. head: ln_f (2*C) + lm_head (C * V + V)
    # 合計を返す
    pass
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：パラメータを「入口（`tok_emb + pos_emb`）」「1ブロック（`ln1 + ln2 + attn + ffwd`）× `n_layer`」「出口（`ln_f + lm_head`）」の 3 部分に分けて足し合わせます。
- **ヒント 2**：`emb = V * C + block_size * C`、`per_block = 4*C + (4*C*C + C) + (8*C*C + 5*C) = 12*C*C + 10*C`、`head = 2*C + C*V + V` を合計して返しましょう。

**解答例：**
```python
def count_gpt_params(V, block_size, n_embd, n_layer):
    C = n_embd
    emb = V * C + block_size * C
    per_block = 12 * C * C + 10 * C
    head = 2 * C + C * V + V
    return emb + n_layer * per_block + head
```

**検算テスト（Colabで実行して確認）：**
```python
# 第12章本文の実測値と完全一致するか確認
assert count_gpt_params(2061, 64, 64, 2) == 369677
# 第13章ミニGPT（約175万パラメータ）
assert count_gpt_params(3665, 128, 128, 4) == 1750097
# 小さな検算（V=10, T=4, C=8, L=1）
# emb = 80+32=112, block = 12*64+80=848, head = 16+80+10=106 -> 1066
assert count_gpt_params(10, 4, 8, 1) == 1066
print("ALL PASSED")
```

**考え方**：モデルを実行する前に設定値だけでパラメータ総数（＝おおよそのVRAM必要量）を1個単位で正確に見積もれるようになると、LLM研究の設計力が一気に上がります。

1. 埋め込み：`tok_emb`（$VC$）＋ `pos_emb`（$TC$）。
1. 1ブロック：`ln1, ln2`（$4C$）＋ `attn`（$4C^2 + C$）＋ `ffwd`（$8C^2 + 5C$）＝ $12C^2 + 10C$。
1. 出口：`ln_f`（$2C$）＋ `lm_head`（$CV + V$）。
1. 合計：`emb + n_layer * per_block + head`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/12-transformer.html#ex-ch12-q12

</details>

*(この節の追加問題 `ch12-q13` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/12-transformer.html で自動採点・復習できます)*

## 1バッチで丸暗記させるテスト

第10章で紹介した研究者の口ぐせ「まず小さなデータで過学習させてみよ」を実行します。同じ1バッチだけを300回学習させて、丸暗記できるか試します。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
torch.manual_seed(0)
model = GPT(V, block_size=64, n_embd=64, n_head=4, n_layer=2, dropout=0.0)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
x, y = get_batch(train_data, 8, 64, torch.Generator().manual_seed(0))   # この1バッチだけ
for step in range(301):
    _, loss = model(x, y)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 100 == 0:
        print(f"ステップ {step}：損失 {loss.item():.3f}")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
ステップ 0：損失 7.656
ステップ 100：損失 0.025
ステップ 200：損失 0.012
ステップ 300：損失 0.009
```

</details>

損失がほぼ0になりました。モデルは、このバッチの文章を完全に丸暗記しました。**丸暗記ができる＝モデルと学習ループにバグがない**、ことの確認です。もし下がらなかったら、どこかの部品のつなぎ方を間違えています（`ModuleList` を普通のリストにしてしまった、など）。

<details><summary><b>基礎の補足：なぜ「1バッチ丸暗記テスト」がML研究最強のデバッグ技なのか？</b>（クリックで開く）</summary>

深層学習のコードは、バグがあっても**エラーを出さずにそれっぽく動き続けてしまう**ことがよくあります（サイレントバグ）。数時間GPUを回したあとに「損失が下がらない！ 実は `nn.ModuleList` ではなく普通のリストだった」と気づくのは最悪のパターンです。

そこで研究者は、本学習の前に必ず次の **1バッチ丸暗記テスト（overfit-one-batch sanity test）** を数秒だけ走らせます。

1. `dropout = 0.0` にする（過学習を邪魔するブレーキを外す）
2. 固定の1バッチ `(x, y)` だけを何百回も繰り返して学習させる
3. **損失がほぼ 0（`0.01` 未満）まで下がるか確認する**

もし下がらなければ、順伝播・正解のずらし方・`ModuleList` 登録・`opt.zero_grad()`・学習率のどこかに必ずバグがあります。逆にほぼ 0 まで落ちれば「少なくとも学習の配線は100%正常」と安心して本学習へ進めます。

</details>

#### 📝 確認問題（`ch12-q14` · Lv.3 · 1バッチ丸暗記（過学習）テストが最強のデバッグ技である理由）

新しいモデルを実装したとき、いきなり全データで長時間学習させるのではなく、最初に「**同じ1バッチだけを（`dropout=0.0` で）丸暗記させるテスト**」を行うのはなぜですか？

- **a.** 最初に 1 バッチを完全に丸暗記した重みを初期値として本学習へ引き継ぐことで、最終的な検証損失を確実に `0.0` 付近まで下げられるから
- **b.** ランダムに選んだ 1 バッチだけを徹底的に暗記させておけば、全訓練データで長時間学習させた場合と同等の汎化性能を数秒で獲得できるから
- **c.** PyTorch のオプティマイザ `AdamW` の内部仕様として、事前に 1 バッチで勾配を初期化しておかないと本学習のステップが実行できないから
- **d.** たった 1 バッチすら損失ほぼ 0 に暗記できないならモデル構造や学習ループにバグがあると数秒で分かり、逆に暗記できれば基本配線が正しいことを確認できるから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：何時間も GPU を回したあとに「`nn.ModuleList` にし忘れていた！」と気づく悲劇を、最初の数秒で見抜くためのテストです。
- **ヒント 2**：最小データ（1バッチ）を丸暗記できることはモデルと学習ループに致命的バグがないための最低条件（健全性チェック）です。
- **正解**：`d`

**考え方**：過学習できる能力は、モデルが正しく学習できるための最低条件です。1バッチテストで損失が0付近まで落ちない場合、`ModuleList` 漏れ・`zero_grad` 漏れ・正解のずらしミス・学習率異常などのバグを即座に特定できます。

1. `dropout=0.0` でノイズを切り、固定の1バッチ `(x, y)` を300ステップ繰り返す。
1. 損失が `7.656 → 0.009` のように0近くまで落ちれば、基本配線はOK。
1. もし下がらなければ、本格学習に入る前にバグを直す。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/12-transformer.html#ex-ch12-q14

</details>

## 『こころ』で少しだけ学習させる

本格的な学習は第13章でGPUを使ってやります。ここでは、CPUで1500ステップだけ学習させて、ちゃんと学べることを確かめます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
@torch.no_grad()
def val_loss(model, T):
    model.eval()
    starts = range(0, len(val_data) - T - 1, T)
    xs = torch.stack([val_data[i:i + T] for i in starts])
    ys = torch.stack([val_data[i + 1:i + T + 1] for i in starts])
    loss = model(xs, ys)[1].item()
    model.train()
    return loss

torch.manual_seed(0)
g = torch.Generator().manual_seed(0)
model = GPT(V, block_size=64, n_embd=64, n_head=4, n_layer=2, dropout=0.1)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
for step in range(1501):
    x, y = get_batch(train_data, 32, 64, g)
    _, loss = model(x, y)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 500 == 0:
        print(f"ステップ {step:4d}：訓練 {loss.item():.3f}  検証 {val_loss(model, 64):.3f}")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
ステップ    0：訓練 7.646  検証 7.495
ステップ  500：訓練 3.092  検証 3.316
ステップ 1000：訓練 3.041  検証 3.148
ステップ 1500：訓練 2.741  検証 3.097
```

</details>

1500ステップ、たった37万個のパラメータで、第11章の1ヘッドモデル（28万個、2000ステップで3.31）より良い検証損失になりました。まだ下がり続けています。続きは、第13章でGPUを使って思う存分やりましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
model.eval()
torch.manual_seed(0)
start = torch.tensor([[stoi[c] for c in "私はその人を"]])
out = model.generate(start, max_new_tokens=80, temperature=0.8)
print("".join(itos[i] for i in out[0].tolist()))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
私はその人を選んな一人の間に見て、今、念とうとうところでした。私は一種を語る時々らなくなった。父はまた私を欺くより返して来た。私に告げた一人にする親類を利き事があった。この
```

</details>

## このコードの行き先

この章で書いた5つのクラス（`Head`、`MultiHeadAttention`、`FeedForward`、`Block`、`GPT`）は、教科書のリポジトリに **`code/minigpt.py`** として、まったく同じ内容で置いてあります。第13章からは、毎回このファイルをダウンロードして使います。あなたが書いたものと同じものです。自信を持って `import` してください。

<details><summary><b>もっと深く：本物のGPT-2との違い</b>（クリックで開く）</summary>

この章のGPTと、OpenAIが2019年に公開したGPT-2の構造は、ほとんど同じです。違いは主に次の3つで、第15章で本物のGPT-2の重みを読み込むときに、ここを合わせます。

- GPT-2は、Q・K・Vを作る3つの全結合層を1つにまとめている（計算が速い）。数学的には同じ
- GPT-2は、Q・K・Vの全結合層にもバイアスがある
- GPT-2は、出力層の重みを文字の埋め込みの表と**共有**している（weight tying。パラメータの節約）

大きさは、GPT-2の一番小さい版で12層・12ヘッド・768次元、約1億2400万パラメータです。

</details>

> **⚠️ つまずき注意：よくあるつまずき**
>
> - **`RuntimeError: mat1 and mat2 shapes cannot be multiplied (…x60 and 64x64)`**：`n_embd` が `n_head` で割り切れていません。たとえば `n_embd=64, n_head=5` だと、1つのヘッドの大きさが `64 // 5 = 12` になり、5つのヘッドをつなげても `5 × 12 = 60` で、64に戻りません。64と4、128と4、384と6のように、割り切れる組み合わせにしましょう。
> - **損失がまったく下がらない**：`self.heads` を `nn.ModuleList` ではなく普通のリストにしていませんか？ `print(model)` で部品の一覧を表示して、ヘッドが入っているか確認しましょう。
> - **`IndexError: index out of range in self`**：`block_size` より長い文字列を `model(…)` に入れています。`generate` は自動で切り詰めますが、自分で `model(x)` を呼ぶときは注意。

#### 📝 確認問題（`ch12-q15` · Lv.4 · 自作ミニGPTの全体設計と本物のGPT-2との3つの違い）

第12章で作った `minigpt.py`（`Head`・`MultiHeadAttention`・`FeedForward`・`Block`・`GPT`）の構造は本物の GPT-2 とほとんど同じです。本物の GPT-2 との主な3つの違いを挙げ、なぜ第12章のミニGPTではその簡易版を選んだのか説明してください。

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：本文の「もっと深く：本物の GPT-2 との違い」で挙げた 3 つの実装上の違い（QKV の書き方・バイアス・出力層の重み）を思い出しましょう。
- **ヒント 2**：①Q・K・V の 3 層を 1 つの線形層へ統合、②Q・K・V にもバイアスあり、③`tok_emb` と `lm_head` の重み共有（weight tying）の 3 点と、ミニ GPT が可読性を優先した理由をまとめます。
- **解答の目安**：本物の GPT-2 との主な違いは、(1) Q・K・V を作る3つの線形層を1つの大きな線形層にまとめて高速化している点（数学的には同一）、(2) Q・K・V の線形層にもバイアス（bias）がある点、(3) 出力層 `lm_head` の重みを文字埋め込み `tok_emb` と共有（weight tying）してパラメータを節約している点、の3つです。第12章のミニGPTでは「計算の速さや細かな最適化」よりも「Q・K・V や各部品の役割が一目で読めるわかりやすさ」を優先して素直に書いており、数学的な本質は GPT-2 と同じです（第15章でこの3点を合わせて本物のGPT-2重みを読み込みます）。

**考え方**：たった120行の `code/minigpt.py` が、本質的に OpenAI の GPT-2 と同じアーキテクチャであることを理解すれば、現代LLMの構造はもうブラックボックスではありません。

1. 違い①：Q・K・V の3層を1つの線形層に統合（高速化・数学的には同じ）。
1. 違い②：Q・K・V の線形層にもバイアスあり。
1. 違い③：`tok_emb` と `lm_head` の重み共有（weight tying）。
1. ミニGPTでは学習者が1行ずつ構造を理解しやすいように簡潔に書いている。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/12-transformer.html#ex-ch12-q15

</details>

**✅ できたらチェック**

- [ ] マルチヘッドAttentionで、ヘッドの出力をつなげて元の幅に戻せた
- [ ] フィードフォワード層の役割（各位置で考える）を説明できる
- [ ] 残差接続が「情報と勾配の高速道路」である理由を説明できる
- [ ] GPTクラスを組み立て、パラメータの内訳を調べた
- [ ] 1バッチの丸暗記テストで、損失がほぼ0になるのを確かめた
- [ ] 『こころ』で少し学習させ、文章を生成した

> **📝 まとめ**
>
> - **マルチヘッドAttention**：小さなヘッドを並べて、違う観点で過去を見る（話し合い）
> - **フィードフォワード**：各位置で独立に考える小さなMLP（4倍に広げてGELU）
> - **残差接続** `x = x + f(x)`：情報と勾配の高速道路。深い層を可能にする
> - **Pre-LN**：各部品の前でLayerNorm
> - GPT ＝ 埋め込み ＋ Block×N ＋ LayerNorm ＋ 出力層。これが `code/minigpt.py`

**ボンゴ先生**：組み立て完了！ キミのGPTは、まだCPUで少し走っただけの子馬だ。次の章では、GPUという牧場で、漱石の小説を100万字読ませて、思い切り走らせる。どこまで速くなるか、楽しみだね。

## 学習のふりかえり・次のステップ

- **困ったとき・エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **忘れる前に定着を確認**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html) で確認問題・類題・7段階の進捗を記録できます
- **この章のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/12-transformer.html
- **次の章**：[第13章 ミニGPTを鍛える：GPUで本格学習](https://sakunokyumei.github.io/llm-research-textbook/13-train-gpt.html)